# Run benchmark test for NPE

In [1]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import warnings
from sbi.inference import NPE_C
from sbi.diagnostics import run_sbc, check_sbc
import sys
sys.path.append('../pysimARG')

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load genome data and clonal tree.

In [2]:
clonal_edge = np.loadtxt("../data/ClonalOrigin/clonal_edge.csv", delimiter=",", dtype=float)
clonal_node_height = np.loadtxt("../data/ClonalOrigin/clonal_node_height.csv", delimiter=",", dtype=float)

In [3]:
clonal_edge, clonal_node_height

(array([[3.10000000e+01, 1.40000000e+01, 1.80220314e-03],
        [3.10000000e+01, 2.90000000e+01, 1.80220314e-03],
        [3.20000000e+01, 2.80000000e+01, 5.97471972e-03],
        [3.20000000e+01, 2.60000000e+01, 5.97471972e-03],
        [3.30000000e+01, 3.00000000e+00, 8.25024952e-03],
        [3.30000000e+01, 2.70000000e+01, 8.25024952e-03],
        [3.40000000e+01, 2.20000000e+01, 1.30036692e-02],
        [3.40000000e+01, 1.00000000e+01, 1.30036692e-02],
        [3.50000000e+01, 1.10000000e+01, 2.34636374e-02],
        [3.50000000e+01, 1.80000000e+01, 2.34636374e-02],
        [3.60000000e+01, 3.30000000e+01, 1.80941931e-02],
        [3.60000000e+01, 2.10000000e+01, 2.63444427e-02],
        [3.70000000e+01, 1.70000000e+01, 3.56575696e-02],
        [3.70000000e+01, 6.00000000e+00, 3.56575696e-02],
        [3.80000000e+01, 1.20000000e+01, 3.73864931e-02],
        [3.80000000e+01, 3.20000000e+01, 3.14117734e-02],
        [3.90000000e+01, 3.00000000e+01, 3.85383268e-02],
        [3.900

In [4]:
drop_col = range(16, 32)

In [5]:
theta_test = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")

theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = np.delete(x_test, drop_col, axis=1)
x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

print(theta_test.shape, x_test.shape)

torch.Size([1000, 3]) torch.Size([1000, 30])


In [6]:
nan_row = np.where(np.isnan(x_test_numpy))[0]
nan_row

array([865, 899])

In [7]:
theta_test = theta_test[~np.isnan(x_test_numpy).any(axis=1)]
x_test = x_test[~np.isnan(x_test_numpy).any(axis=1)]
theta_test.shape, x_test.shape

(torch.Size([998, 3]), torch.Size([998, 30]))

In [8]:
theta_test_numpy = theta_test.cpu().numpy()
x_test_numpy = x_test.cpu().numpy()
theta_test_numpy.shape, x_test_numpy.shape

((998, 3), (998, 30))

In [9]:
theta1 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")

theta2 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

theta3 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta3.csv', delimiter=",")
x3 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x3.csv', delimiter=",")

theta4 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/theta4.csv', delimiter=",")
x4 = np.loadtxt('../data/ClonalOrigin/rho_and_theta/x4.csv', delimiter=",")

x = np.vstack([x1, x2, x3, x4])
x = np.delete(x, drop_col, axis=1)
theta = np.vstack([theta1, theta2, theta3, theta4])

print(theta.shape, x.shape)

(50000, 3) (50000, 30)


In [10]:
theta = torch.tensor(theta, device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = torch.tensor(x, device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

## Benchmark setting

In [11]:
budgets = [200, 500, 1000, 2000, 5000, 10000, 50000]

In [12]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    print("Kolmogorov-Smirnov Test Results (SBC):")
    print("-" * 40)

    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)

        print(f"Parameter:        {parameter_labels[dim]}")
        print(f"KS Statistic (D): {ks_stat:.4f}")
        print(f"p-value:          {p_value:.4e}")

        if p_value < 0.05:
            print("Status: MISCALIBRATED (reject null)")
        else:
            print("Status: CALIBRATED (fail to reject null)")
        print("-" * 40)
    
    return ks_results, p_values

In [13]:
def multidim_coverage95(theta_est_post, theta_test_numpy):
    print("\nJoint 3D Coverage Metric:")
    print("-" * 40)

    num_test_samples = theta_test_numpy.shape[0]
    joint_covered_count = 0

    for i in range(num_test_samples):
        samples = theta_est_post[i]  # Shape: (1000, 3)
        truth = theta_test_numpy[i]  # Shape: (3,)

        mean_vec = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)
        
        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, skipping.")
            continue

        diff_samples = samples - mean_vec
        dist_samples = np.sum(np.dot(diff_samples, inv_cov) * diff_samples, axis=1)

        diff_truth = truth - mean_vec
        dist_truth = np.dot(np.dot(diff_truth, inv_cov), diff_truth)

        threshold_95 = np.percentile(dist_samples, 95)

        if dist_truth <= threshold_95:
            joint_covered_count += 1

    joint_coverage_95 = joint_covered_count / num_test_samples

    print(f"Target Joint Coverage: 95.0%")
    print(f"Actual Joint Coverage: {joint_coverage_95 * 100:.1f}%")
    print("-" * 40)

    return joint_coverage_95

In [14]:
def marginal_coverage95(theta_est_post, theta_test_numpy):
    print("\nMarginal 1D Coverage Metrics (95% Interval):")
    print("-" * 40)
    
    lower_bounds = np.percentile(theta_est_post, 2.5, axis=1)
    upper_bounds = np.percentile(theta_est_post, 97.5, axis=1)
    is_covered = (theta_test_numpy >= lower_bounds) & (theta_test_numpy <= upper_bounds)
    marginal_coverages = np.mean(is_covered, axis=0)

    for d, coverage in enumerate(marginal_coverages):
        print(f"Dimension {d} Coverage: {coverage * 100:.1f}%")
    print("-" * 40)
    
    return marginal_coverages

In [16]:
def L2_error(theta_est_post, theta_test_numpy):
    post_mean_error = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        post_mean = np.mean(theta_est_post[i, :, :], axis=0)
        error = post_mean - theta_test_numpy[i, :]
        post_mean_error[i] = np.linalg.norm(error)
    return post_mean_error

In [17]:
def mahalanobis_error(theta_est_post, theta_test_numpy):
    maha_errors = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        samples = theta_est_post[i]
        truth = theta_test_numpy[i]
        post_mean = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)

        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, returning NaN.")
            continue
        
        diff = post_mean - truth
        maha_dist_sq = np.dot(np.dot(diff, inv_cov), diff)
        maha_errors[i] = np.sqrt(maha_dist_sq)
    return maha_errors

## NPE

In [18]:
npe_sbc_D = np.full((3, len(budgets)), np.nan)
npe_sbc_p_values = np.full((3, len(budgets)), np.nan)
npe_coverage_results = np.full((2, len(budgets)), np.nan)
npe_mean_error_results = np.full((theta_test.shape[0], len(budgets), 2), np.nan)

npe_sbc_D.shape, npe_sbc_p_values.shape, npe_coverage_results.shape, npe_mean_error_results.shape

((3, 7), (3, 7), (2, 7), (998, 7, 2))

In [19]:
seed = 100
num_posterior_samples=1000
learning_rate = 0.0005

In [20]:
i = 0
torch.manual_seed(seed)
np.random.seed(seed)
print("-" * 50)
n_sim = budgets[i]
x_train = x[:n_sim]
theta_train = theta[:n_sim]

print(f"\nTraining NPE with {n_sim} simulations")

inference_benchmark = NPE_C(density_estimator="nsf", device=torch_device)
density_estimator_benchmark = inference_benchmark.append_simulations(theta_train, x_train).train(
    max_num_epochs=500, learning_rate=learning_rate
)
posterior_benchmark = inference_benchmark.build_posterior(density_estimator_benchmark)

print(f"Sampling posterior for NPE, n={n_sim}")

theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
    theta_post = posterior_benchmark.sample((num_posterior_samples,), x=x_test[j, :],
                                            show_progress_bars=False)
    theta_est_post[j, :, :] = theta_post.detach().numpy()

print(f"Calculating metrics for NPE, n={n_sim}")

sbc_results = []
parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
ranks, dap_samples = run_sbc(
    theta_test,
    x_test,
    posterior_benchmark,
    num_posterior_samples=num_posterior_samples,
    use_batched_sampling=False
)

ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
coverage_95_with_L = multidim_coverage95(theta_est_post, theta_test_numpy)
coverage_95_without_L = multidim_coverage95(theta_est_post[:, :, :2], theta_test_numpy[:, :2])
mean_error_with_L = L2_error(theta_est_post, theta_test_numpy)
mean_error_without_L = L2_error(theta_est_post[:, :, :2], theta_test_numpy[:, :2])

npe_sbc_D[:, i] = ks_results
npe_sbc_p_values[:, i] = p_values
npe_coverage_results[0, i] = coverage_95_with_L
npe_coverage_results[1, i] = coverage_95_without_L
npe_mean_error_results[:, i, 0] = mean_error_with_L
npe_mean_error_results[:, i, 1] = mean_error_without_L

print("-" * 50)

--------------------------------------------------

Training NPE with 200 simulations
 Neural network successfully converged after 185 epochs.Sampling posterior for NPE, n=200


Sampling posterior: 100%|██████████| 998/998 [00:15<00:00, 62.71it/s]


Calculating metrics for NPE, n=200


Calculating ranks for 998 SBC samples: 100%|██████████| 998/998 [00:00<00:00, 13345.48it/s]


Kolmogorov-Smirnov Test Results (SBC):
----------------------------------------
Parameter:        for $\rho_s$
KS Statistic (D): 0.0603
p-value:          1.3389e-03
Status: MISCALIBRATED (reject null)
----------------------------------------
Parameter:        for $\theta_s$
KS Statistic (D): 0.0374
p-value:          1.1887e-01
Status: CALIBRATED (fail to reject null)
----------------------------------------
Parameter:        for L
KS Statistic (D): 0.1025
p-value:          1.4059e-09
Status: MISCALIBRATED (reject null)
----------------------------------------

Joint 3D Coverage Metric:
----------------------------------------
Target Joint Coverage: 95.0%
Actual Joint Coverage: 96.4%
----------------------------------------

Joint 3D Coverage Metric:
----------------------------------------
Target Joint Coverage: 95.0%
Actual Joint Coverage: 96.4%
----------------------------------------
--------------------------------------------------


In [ ]:
for i in range(len(budgets)):
    torch.manual_seed(seed)
    np.random.seed(seed)
    print("-" * 50)
    n_sim = budgets[i]
    x_train = x[:n_sim]
    theta_train = theta[:n_sim]

    print(f"\nTraining NPE with {n_sim} simulations")
    
    inference_benchmark = NPE_C(density_estimator="nsf", device=torch_device)
    density_estimator_benchmark = inference_benchmark.append_simulations(theta_train, x_train).train(
        max_num_epochs=500, learning_rate=learning_rate
    )
    posterior_benchmark = inference_benchmark.build_posterior(density_estimator_benchmark)

    print(f"Sampling posterior for NPE, n={n_sim}")

    theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
    for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
        theta_post = posterior_benchmark.sample((num_posterior_samples,), x=x_test[j, :],
                                                show_progress_bars=False)
        theta_est_post[j, :, :] = theta_post.detach().numpy()
    
    print(f"Calculating metrics for NPE, n={n_sim}")

    sbc_results = []
    parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
    ranks, dap_samples = run_sbc(
        theta_test,
        x_test,
        posterior_benchmark,
        num_posterior_samples=num_posterior_samples,
        use_batched_sampling=False
    )

    ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
    coverage_95_with_L = multidim_coverage95(theta_est_post, theta_test_numpy)
    coverage_95_without_L = multidim_coverage95(theta_est_post[:, :, :2], theta_test_numpy[:, :2])
    mean_error_with_L = L2_error(theta_est_post, theta_test_numpy)
    mean_error_without_L = L2_error(theta_est_post[:, :, :2], theta_test_numpy[:, :2])

    npe_sbc_D[:, i] = ks_results
    npe_sbc_p_values[:, i] = p_values
    npe_coverage_results[0, i] = coverage_95_with_L
    npe_coverage_results[1, i] = coverage_95_without_L
    npe_mean_error_results[:, i, 0] = mean_error_with_L
    npe_mean_error_results[:, i, 1] = mean_error_without_L

    print("-" * 50)

In [ ]:
np.savetxt("../data/benchmark/npe_sbc_D.csv", npe_sbc_D, delimiter=",")
np.savetxt("../data/benchmark/npe_sbc_p_values.csv", npe_sbc_p_values, delimiter=",")
np.savetxt("../data/benchmark/npe_coverage_results.csv", npe_coverage_results, delimiter=",")
np.save("../data/benchmark/npe_mean_error_results.npy", npe_mean_error_results)